In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy import stats
from nba_api.stats.endpoints import leaguedashteamstats
from datetime import datetime

pd.set_option('display.max_columns', None)

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

In [ ]:
from src.points_model import PointsPropModel
from src.points_model.utils import generate_synthetic_game_logs

# Generate test data
game_logs = generate_synthetic_game_logs(n_players=50, games_per_player=30)

# Test your model
model = PointsPropModel()
model.fit(game_logs)

# Test projections
projection = model.project_points(player_id=1, is_b2b=False)
pd.DataFrame(projection)

In [17]:
from src.points_model.utils import generate_synthetic_game_logs

# Generate synthetic data with default parameters
game_logs = generate_synthetic_game_logs()

# Generate 50 players with 30 games each
game_logs = generate_synthetic_game_logs(n_players=50, games_per_player=30)
print(f"Created {len(game_logs)} game logs for {game_logs['player_id'].nunique()} players")


Created 1500 game logs for 50 players


In [10]:
def get_game_spread(team_abbrev, current_date, team_lines_dir='data/raw/team_lines'):
    """
    Get the spread for a team's game on a given date.
    """
    # Convert date to file format (YYYYMMDD)
    date_str = current_date.replace('-', '')
    
    # Find all files matching the date pattern
    team_lines_path = Path(team_lines_dir)
    pattern = f'NBA_{date_str}_*.json'
    matching_files = list(team_lines_path.glob(pattern))
    
    if not matching_files:
        return None
    
    # Get the latest file (by modification time, or by time in filename)
    # Option 1: By modification time (most recent scrape)
    latest_file = max(matching_files, key=lambda p: p.stat().st_mtime)
    
    # Option 2: By time in filename (if you prefer)
    # latest_file = max(matching_files, key=lambda p: int(p.stem.split('_')[-1]) if p.stem.split('_')[-1].isdigit() else 0)
    
    try:
        with open(latest_file, 'r') as f:
            games_data = json.load(f)
        
        # Map team abbreviations to full names
        team_name_map = {
            'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'BKN': 'Brooklyn Nets',
            'CHA': 'Charlotte Hornets', 'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
            'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'DET': 'Detroit Pistons',
            'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
            'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers', 'MEM': 'Memphis Grizzlies',
            'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
            'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks', 'OKC': 'Oklahoma City Thunder',
            'ORL': 'Orlando Magic', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
            'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs',
            'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards'
        }
        
        team_full_name = team_name_map.get(team_abbrev)
        if not team_full_name:
            return None
        
        # Find the game with this team
        for game in games_data:
            is_home = game['home_team'] == team_full_name
            is_away = game['away_team'] == team_full_name
            
            if is_home or is_away:
                # Get spread from first bookmaker (or average across bookmakers)
                for bookmaker in game['bookmakers']:
                    for market in bookmaker['markets']:
                        if market['market_key'] == 'spreads':
                            for outcome in market['outcomes']:
                                if outcome['name'] == team_full_name:
                                    spread = outcome['point']
                                    # Spread is from team's perspective
                                    # Negative = favored (expected to win by that amount)
                                    # Positive = underdog (expected to lose by that amount)
                                    return spread
        return None
    except Exception as e:
        return None

def calculate_blowout_prob_from_spread(spread):
    """
    Calculate blowout probability from spread.
    
    Blowout = |actual margin| > 20
    Using spread as expected margin, calculate probability of blowout.
    """
    if spread is None:
        return 0.15  # Default fallback
    
    # Expected margin from team's perspective
    # If spread is -4.5, team is expected to win by 4.5
    # If spread is +4.5, team is expected to lose by 4.5
    expected_margin = -spread  # Flip sign: negative spread = positive margin
    
    # Typical NBA game margin std dev is ~12 points
    margin_std = 12.0
    
    # Probability of blowout = P(|margin| > 20)
    # This is P(margin > 20) + P(margin < -20)
    prob_win_blowout = 1 - stats.norm.cdf(20, expected_margin, margin_std)
    prob_loss_blowout = stats.norm.cdf(-20, expected_margin, margin_std)
    blowout_prob = prob_win_blowout + prob_loss_blowout
    
    # Clamp between reasonable bounds
    return max(0.05, min(0.40, blowout_prob))

### Generate different scenrios for a certain player

In [18]:
player_id = 2544
market_line = 20.5
market_juice = -110

In [19]:
# =============================================================================
# PLAYER SCENARIO ANALYSIS FOR UPCOMING GAME
# =============================================================================

from src.points_model import PointsPropModel
from src.utils.helper_functions import findOpp
from datetime import datetime

# Load your real data (same as production)
s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv')

def parse_minutes(min_str):
    if pd.isna(min_str): return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

s26_prepped = s26.copy()
s26_prepped['minutes'] = s26_prepped['MIN'].apply(parse_minutes)
s26_prepped = s26_prepped.rename(columns={
    'PLAYER_ID': 'player_id', 'PLAYER_NAME': 'player_name',
    'FGA': 'fga', 'FG3A': 'fg3a', 'FTA': 'fta',
    'FGM': 'fgm', 'FG3M': 'fg3m', 'FTM': 'ftm',
    'PTS': 'pts', 'PLUS_MINUS': 'margin'
})

# Get team mapping
team_abbrev_to_id = s26_prepped.groupby('TEAM_ABBREVIATION')['TEAM_ID'].first().to_dict()
name_to_team = s26_prepped.groupby('player_name')['TEAM_ABBREVIATION'].last().to_dict()

# Fit model
model = PointsPropModel(min_edge=0.02, min_confidence=0.52)
model.fit(s26_prepped)

# Players info
player_name = s26_prepped[s26_prepped['player_id'] == player_id]['player_name'].iloc[0] if len(s26_prepped[s26_prepped['player_id'] == player_id]) > 0 else "LeBron James"
current_date = datetime.now().strftime('%Y-%m-%d')

# Get actual game info for today
player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
is_b2b_actual = False
if not player_games.empty:
    latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
    current_date_dt = pd.to_datetime(current_date)
    days_since_last_game = (current_date_dt - latest_game_date).days
    is_b2b_actual = (days_since_last_game == 1)

# Get opponent info
opp_abbrev, home_flag = findOpp(player_name, s26, current_date)
opp_team_id = None
if opp_abbrev and opp_abbrev in team_abbrev_to_id:
    opp_team_id = int(team_abbrev_to_id[opp_abbrev])

# Get actual opponent stats (if available)
opp_pace_actual = None
opp_drtg_actual = None
if opp_team_id:
    opp_stats = model.matchup_adjuster.get_team_stats(opp_team_id)
    if opp_stats is not None:
        opp_pace_actual = opp_stats.get('PACE', 100.0)
        opp_drtg_actual = opp_stats.get('DEF_RATING', 114.0)

# Get actual spread for blowout calculation
player_team_abbrev = name_to_team.get(player_name)
spread_actual = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
blowout_prob_actual = calculate_blowout_prob_from_spread(spread_actual)

# Define scenarios
scenarios = []

# Scenario 1: ACTUAL GAME CONDITIONS (baseline)
scenarios.append({
    'name': '🎯 ACTUAL GAME CONDITIONS',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': f'B2B: {is_b2b_actual}, Blowout: {blowout_prob_actual:.1%}, Opp: {opp_abbrev or "Unknown"}'
})

# Scenario 2: If it was a back-to-back (even if it's not)
scenarios.append({
    'name': '⚠️ IF BACK-TO-BACK',
    'is_b2b': True,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': 'Same game but on B2B'
})

# Scenario 3: Higher blowout risk
scenarios.append({
    'name': '📉 HIGH BLOWOUT RISK',
    'is_b2b': is_b2b_actual,
    'blowout_prob': min(0.40, blowout_prob_actual + 0.20),
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': 'Increased blowout probability'
})

# Scenario 4: Fast-paced opponent
scenarios.append({
    'name': '⚡ FAST-PACED OPPONENT',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': (opp_pace_actual or 100.0) + 8,  # +8 possessions
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.0,
    'description': 'More possessions = more opportunities'
})

# Scenario 5: Weak defense opponent
scenarios.append({
    'name': '🛡️ WEAK DEFENSE',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': (opp_drtg_actual or 114.0) + 4,  # Higher DRTG = worse defense
    'usage_adjustment': 1.0,
    'description': 'Easier scoring opportunities'
})

# Scenario 6: Strong defense opponent
scenarios.append({
    'name': '🛡️ STRONG DEFENSE',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': (opp_drtg_actual or 114.0) - 4,  # Lower DRTG = better defense
    'usage_adjustment': 1.0,
    'description': 'Tougher matchup'
})

# Scenario 7: Star teammate out (usage boost)
scenarios.append({
    'name': '⭐ STAR TEAMMATE OUT',
    'is_b2b': is_b2b_actual,
    'blowout_prob': blowout_prob_actual,
    'opp_pace': opp_pace_actual or 100.0,
    'opp_drtg': opp_drtg_actual or 114.0,
    'usage_adjustment': 1.15,  # 15% usage boost
    'description': 'More touches and shots'
})

# Scenario 8: Best case scenario
scenarios.append({
    'name': '🚀 BEST CASE',
    'is_b2b': False,
    'blowout_prob': 0.05,
    'opp_pace': (opp_pace_actual or 100.0) + 8,
    'opp_drtg': (opp_drtg_actual or 114.0) + 4,
    'usage_adjustment': 1.15,
    'description': 'Rested + fast pace + weak defense + usage boost'
})

# Scenario 9: Worst case scenario
scenarios.append({
    'name': '📉 WORST CASE',
    'is_b2b': True,
    'blowout_prob': 0.35,
    'opp_pace': (opp_pace_actual or 100.0) - 5,
    'opp_drtg': (opp_drtg_actual or 114.0) - 4,
    'usage_adjustment': 0.9,
    'description': 'B2B + blowout risk + slow pace + strong defense'
})

# Generate projections for all scenarios
results = []
for scenario in scenarios:
    projection = model.project_points(
        player_id=player_id,
        is_b2b=scenario['is_b2b'],
        blowout_prob=scenario['blowout_prob'],
        opp_pace=scenario['opp_pace'],
        opp_drtg=scenario['opp_drtg'],
        usage_adjustment=scenario['usage_adjustment'],
        opp_team_id=opp_team_id if scenario['opp_pace'] == (opp_pace_actual or 100.0) else None
    )
    
    if projection:
        results.append({
            'scenario': scenario['name'],
            'expected_points': projection['expected_points'],
            'std': projection['std'],
            'lower_90': projection['lower_90'],
            'upper_90': projection['upper_90'],
            'minutes': projection['components']['minutes']['expected'],
            'fga': projection['components']['volume']['fga'],
            'fg3a': projection['components']['volume']['fg3a'],
            'fta': projection['components']['volume']['fta'],
            'description': scenario.get('description', '')
        })

# Display results
scenarios_df = pd.DataFrame(results)
scenarios_df = scenarios_df.sort_values('expected_points', ascending=False)

print("\n" + "="*100)
print(f"{s26_prepped[s26_prepped['player_id'] == player_id]['player_name'].iloc[0]} (ID: {player_id}) - SCENARIO ANALYSIS FOR UPCOMING GAME")
print(f"Date: {current_date} | Opponent: {opp_abbrev or 'Unknown'}")
print("="*100)
print(scenarios_df[['scenario', 'expected_points', 'std', 'lower_90', 'upper_90', 'minutes', 'fga', 'description']].to_string(index=False))

# Summary statistics
print(f"\n📊 PROJECTION RANGE:")
print(f"   Minimum: {scenarios_df['expected_points'].min():.1f} pts ({scenarios_df.loc[scenarios_df['expected_points'].idxmin(), 'scenario']})")
print(f"   Maximum: {scenarios_df['expected_points'].max():.1f} pts ({scenarios_df.loc[scenarios_df['expected_points'].idxmax(), 'scenario']})")
print(f"   Average: {scenarios_df['expected_points'].mean():.1f} pts")
print(f"   Range: {scenarios_df['expected_points'].max() - scenarios_df['expected_points'].min():.1f} pts")
print(f"   Actual Game: {scenarios_df[scenarios_df['scenario'].str.contains('ACTUAL')]['expected_points'].iloc[0]:.1f} pts")

print(f"\n" + "="*100)
print(f"PROP EVALUATION: Over/Under {market_line} @ {market_juice}")
print("="*100)

for scenario in scenarios:
    evaluation = model.evaluate_prop(
        player_id=player_id,
        market_line=market_line,
        market_juice=market_juice,
        is_b2b=scenario['is_b2b'],
        blowout_prob=scenario['blowout_prob'],
        opp_pace=scenario['opp_pace'],
        opp_drtg=scenario['opp_drtg'],
        usage_adjustment=scenario['usage_adjustment'],
        opp_team_id=opp_team_id if scenario['opp_pace'] == (opp_pace_actual or 100.0) else None
    )
    
    if evaluation:
        edge = evaluation['edge_analysis']
        proj = evaluation['projection']
        print(f"\n{scenario['name']}:")
        print(f"  Projection: {proj['expected_points']:.1f} pts")
        print(f"  P(Over {market_line}): {edge['prob_over']:.1%}")
        print(f"  P(Under {market_line}): {edge['prob_under']:.1%}")
        print(f"  Over Edge: {edge['over_edge']:+.1%}")
        print(f"  Under Edge: {edge['under_edge']:+.1%}")
        print(f"  Recommendation: {edge['recommendation']}")
        print(f"  EV: {edge['expected_value']:+.3f}")
        if edge.get('kelly_bet_size', 0) > 0:
            print(f"  Kelly: {edge['kelly_bet_size']*100:.1f}% ({edge.get('unit_recommendation', 'N/A')})")


LeBron James (ID: 2544) - SCENARIO ANALYSIS FOR UPCOMING GAME
Date: 2025-12-14 | Opponent: PHX
                scenario  expected_points      std  lower_90  upper_90   minutes       fga                                     description
             🚀 BEST CASE        19.929629 3.747736 13.764602 26.094655 33.355000 17.661938 Rested + fast pace + weak defense + usage boost
     ⭐ STAR TEAMMATE OUT        17.849184 3.391017 12.270962 23.427406 33.176016 15.634968                          More touches and shots
   ⚡ FAST-PACED OPPONENT        16.569013 3.173549 11.348525 21.789501 33.176016 14.683709           More possessions = more opportunities
         🛡️ WEAK DEFENSE        16.146877 3.102249 11.043677 21.250077 33.176016 14.143835                    Easier scoring opportunities
🎯 ACTUAL GAME CONDITIONS        15.521030 2.996966 10.591020 20.451039 33.176016 13.595624            B2B: False, Blowout: 11.0%, Opp: PHX
     📉 HIGH BLOWOUT RISK        15.240326 2.949922 10.387705 20.092947